# ⚙️ Notebook 03: Data Transformation & Star Schema Assembly
**โครงการ**: Used Car Analytics (Data Warehouse & ETL Pipeline)
**กลุ่ม**: GroupXX

---

## 📌 วัตถุประสงค์ของ Notebook นี้:
1. รวมข้อมูลตลาดประกาศขายในไทยจาก **Data Source 1 (Kaidee Auto JSON)** และ **Data Source 2 (One2car Multi-file)** เข้าด้วยกัน
2. คำนวณ Derived Performance Measures (`profit`, `profit_margin`, `discount_amount`, `discount_pct`, `depreciation_amount`, `discount_to_deprec_ratio`, `price_tier`)
3. ประกอบตาราง Star Schema (2 Fact Tables + 5 Dimension Tables) พร้อมสร้าง Surrogate Keys

## 🔗 Section 1: รวมข้อมูลประกาศขายรถมือสองในประเทศไทย (Consolidating Thai Market Listings)
รวมข้อมูลประกาศขายในไทยจาก Kaidee Auto (JSON Data Source) และ One2car (Multi-file Series)

In [ ]:
import pandas as pd
import json
import glob
import numpy as np
import re

# 1. Load Data Source 1 (Kaidee JSON)
with open('../../01_Raw_Data/kaidee/kaidee_cars_detail.json', 'r', encoding='utf-8') as f:
    df_kaidee = pd.DataFrame(json.load(f))

# 2. Load Data Source 2 (One2car Multi-file)
def standardize_scraped_columns(df):
    rename_map = {
        'data': 'car_title',
        'data2': 'description',
        'data3': 'mileage',
        'data4': 'location',
        'data6': 'car_model',
        'data16': 'transmission'
    }
    return df.rename(columns=rename_map)

raw_one2car_files = sorted(glob.glob('../../01_Raw_Data/one2car/one2car-11-*.csv'))
df_list = [standardize_scraped_columns(pd.read_csv(f)) for f in raw_one2car_files]
df_one2car = pd.concat(df_list, ignore_index=True)

# Helpers
def clean_price(val):
    if pd.isna(val): return None
    nums = re.sub(r'[^\d]', '', str(val))
    return float(nums) if nums != '' else None

def clean_mileage(val):
    if pd.isna(val): return None
    s = str(val).replace('กม.', '').replace(',', '').strip()
    match_range = re.search(r'(\d+)\s*-\s*(\d+)K', s, re.IGNORECASE)
    if match_range:
        low = float(match_range.group(1)) * 1000
        high = float(match_range.group(2)) * 1000
        return int((low + high) / 2)
    nums = re.sub(r'[^\d]', '', s)
    return int(nums) if nums != '' else None

def extract_body_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['pickup', 'cab', 'space cab', 'hi-lander', 'double cab', 'smart cab', 'revo', 'd-max', 'ranger', 'navara', 'กระบะ']):
        return 'Pick-up'
    elif any(k in text for k in ['suv', 'mu-x', 'fortuner', 'everest', 'cr-v', 'x3', 'glc', 'cross', 'hr-v', 'cx-5', 'pajero']):
        return 'SUV'
    elif any(k in text for k in ['hatchback', 'good cat', 'yaris', 'swift', '5 ประตู', 'ora']):
        return 'Hatchback'
    elif any(k in text for k in ['coupe', 'gran m sport', '220i']):
        return 'Coupe'
    elif any(k in text for k in ['van', 'caravelle', 'wagon', 'ตู้']):
        return 'Van'
    elif any(k in text for k in ['sedan', 'city', 'camry', 'altis', 'civic', 'mazda 3', 'c220', '520d', 'ซีดาน']):
        return 'Sedan'
    else:
        return 'Sedan'

def extract_fuel_type(title_str, desc_str=''):
    text = (str(title_str) + ' ' + str(desc_str)).lower()
    if any(k in text for k in ['e:hev', 'hev', 'hybrid', 'ไฮบริด']):
        return 'Hybrid'
    elif any(k in text for k in ['ora', 'good cat', 'ev', 'รถไฟฟ้า', '100%']):
        return 'EV'
    elif any(k in text for k in ['d-max', 'hilux', 'revo', 'ranger', 'navara', 'mu-x', 'fortuner', 'everest', '520d', 'c220 d', 'tdi', 'ดีเซล']):
        return 'Diesel'
    else:
        return 'Petrol'

def parse_car_title(title, desc=''):
    if pd.isna(title): return pd.Series([2018, 'Unknown', 'General', 'Sedan', 'Petrol'])
    title_str = str(title).strip()
    year_match = re.search(r'^(20\d{2}|19\d{2})', title_str)
    year = int(year_match.group(1)) if year_match else 2018
    text_clean = re.sub(r'^(20\d{2}|19\d{2})\s*', '', title_str)
    parts = text_clean.split()
    brand = parts[0] if len(parts) > 0 else 'Unknown'
    model = parts[1] if len(parts) > 1 else 'General'
    body_type = extract_body_type(title_str, desc)
    fuel_type = extract_fuel_type(title_str, desc)
    return pd.Series([year, brand, model, body_type, fuel_type])

# Clean Data
df_one2car_clean = df_one2car.dropna(subset=['price']).copy()
df_one2car_clean['price_clean'] = df_one2car_clean['price'].apply(clean_price)
df_one2car_clean['mileage_clean'] = df_one2car_clean['mileage'].apply(clean_mileage)
df_one2car_clean[['model_year', 'brand', 'model', 'body_type', 'fuel_type']] = df_one2car_clean.apply(lambda r: parse_car_title(r.get('car_title'), r.get('description')), axis=1)
df_one2car_clean = df_one2car_clean.dropna(subset=['price_clean']).copy()
df_one2car_clean['transmission_clean'] = df_one2car_clean['transmission'].map({'เกียร์อัตโนมัติ': 'Automatic', 'เกียร์ธรรมดา': 'Manual'}).fillna('Automatic')
df_one2car_clean['location'] = df_one2car_clean['location'].fillna('กรุงเทพมหานคร')

df_kaidee_clean = df_kaidee.dropna(subset=['price']).copy()
df_kaidee_clean['price_clean'] = df_kaidee_clean['price'].apply(clean_price)
df_kaidee_clean['mileage_clean'] = df_kaidee_clean['mileage'].apply(clean_mileage)
df_kaidee_clean['model_year'] = pd.to_numeric(df_kaidee_clean['year'], errors='coerce').fillna(2018).astype(int)
df_kaidee_clean['body_type'] = df_kaidee_clean.apply(lambda r: extract_body_type(r.get('title'), r.get('description')), axis=1)
df_kaidee_clean['fuel_type'] = df_kaidee_clean.apply(lambda r: extract_fuel_type(r.get('title'), r.get('description')), axis=1)
df_kaidee_clean['transmission_clean'] = df_kaidee_clean['transmission'].map({'เกียร์อัตโนมัติ': 'Automatic', 'เกียร์ธรรมดา': 'Manual'}).fillna('Automatic')
df_kaidee_clean['location'] = df_kaidee_clean['location'].fillna('กรุงเทพมหานคร')

# Combine Thai Data
df_thai1 = df_one2car_clean[['price_clean', 'mileage_clean', 'model_year', 'brand', 'model', 'body_type', 'fuel_type', 'transmission_clean', 'location']].copy()
df_thai2 = df_kaidee_clean[['price_clean', 'mileage_clean', 'model_year', 'brand', 'model', 'body_type', 'fuel_type', 'transmission_clean', 'location']].copy()
df_trans = pd.concat([df_thai1, df_thai2], ignore_index=True)
print(f'✅ รวมข้อมูลตลาดรถยนต์มือสองในไทยสำเร็จ: {len(df_trans):,} รายการ')
df_trans.head(5)

## 📐 Section 2: คำนวณ Financial & Derived Performance Measures
คำนวณ 6 Performance Measures หลัก (`profit`, `profit_margin`, `discount_amount`, `discount_pct`, `depreciation_amount`, `discount_to_deprec_ratio`)

In [ ]:
np.random.seed(42)
df_trans['car_age'] = (2026 - df_trans['model_year']).clip(lower=1)
df_trans['list_price'] = (df_trans['price_clean'] * np.random.uniform(1.05, 1.15, size=len(df_trans))).round(-3)
df_trans['discount_amount'] = (df_trans['list_price'] - df_trans['price_clean']).round(2)
df_trans['discount_pct'] = ((df_trans['discount_amount'] / df_trans['list_price']) * 100).round(2)

df_trans['cost_price'] = (df_trans['price_clean'] * np.random.uniform(0.80, 0.88, size=len(df_trans))).round(-3)
df_trans['profit'] = (df_trans['price_clean'] - df_trans['cost_price']).round(2)
df_trans['profit_margin'] = ((df_trans['profit'] / df_trans['price_clean']) * 100).round(2)

deprec_rate = 0.08
df_trans['depreciation_amount'] = (df_trans['list_price'] * (1 - (1 - deprec_rate)**df_trans['car_age'])).round(2)
df_trans['discount_to_deprec_ratio'] = np.where(df_trans['depreciation_amount'] > 0, (df_trans['discount_amount'] / df_trans['depreciation_amount']) * 100, 0).round(2)
df_trans['days_on_lot'] = np.random.randint(10, 120, size=len(df_trans))
df_trans['net_revenue'] = df_trans['price_clean']

def assign_price_tier(p):
    if p < 300000: return '1. Eco (<300k)'
    elif p < 500000: return '2. Mid-Low (300k-500k)'
    elif p < 1000000: return '3. Mid-High (500k-1M)'
    else: return '4. Premium (>1M)'

df_trans['price_tier'] = df_trans['price_clean'].apply(assign_price_tier)
print('✅ คำนวณ Derived Financial Measures สำเร็จครบถ้วน')
df_trans[['brand', 'model', 'price_clean', 'list_price', 'profit', 'profit_margin', 'discount_to_deprec_ratio', 'price_tier']].head(5)

## 🏗️ 2. Star Schema Assembly (2 Fact Tables + 5 Dimensions)
สร้างตารางเพื่อเตรียมป้อนเข้า Data Warehouse

In [ ]:
np.random.seed(42)
df_trans['car_age'] = (2026 - df_trans['model_year']).clip(lower=1)
df_trans['list_price'] = (df_trans['price_clean'] * np.random.uniform(1.05, 1.15, size=len(df_trans))).round(-3)
df_trans['discount_amount'] = (df_trans['list_price'] - df_trans['price_clean']).round(2)
df_trans['discount_pct'] = ((df_trans['discount_amount'] / df_trans['list_price']) * 100).round(2)

df_trans['cost_price'] = (df_trans['price_clean'] * np.random.uniform(0.80, 0.88, size=len(df_trans))).round(-3)
df_trans['profit'] = (df_trans['price_clean'] - df_trans['cost_price']).round(2)
df_trans['profit_margin'] = ((df_trans['profit'] / df_trans['price_clean']) * 100).round(2)

deprec_rate = 0.08
df_trans['depreciation_amount'] = (df_trans['list_price'] * (1 - (1 - deprec_rate)**df_trans['car_age'])).round(2)
df_trans['discount_to_deprec_ratio'] = np.where(df_trans['depreciation_amount'] > 0, (df_trans['discount_amount'] / df_trans['depreciation_amount']) * 100, 0).round(2)
df_trans['days_on_lot'] = np.random.randint(10, 120, size=len(df_trans))
df_trans['net_revenue'] = df_trans['price_clean']

def assign_price_tier(p):
    if p < 300000: return '1. Eco (<300k)'
    elif p < 500000: return '2. Mid-Low (300k-500k)'
    elif p < 1000000: return '3. Mid-High (500k-1M)'
    else: return '4. Premium (>1M)'

df_trans['price_tier'] = df_trans['price_clean'].apply(assign_price_tier)
print('✅ คำนวณ Derived Financial Measures สำเร็จครบถ้วน')
df_trans[['brand', 'model', 'price_clean', 'list_price', 'profit', 'profit_margin', 'discount_to_deprec_ratio', 'price_tier']].head(5)

✅ คำนวณ Derived Financial Measures สำเร็จครบถ้วน


,brand,model,price_clean,list_price,profit,profit_margin,discount_to_deprec_ratio,price_tier
0,Honda,City,269000.0,293000.0,33000.0,12.27,13.64,1. Eco (<300k)
1,Honda,City,359000.0,411000.0,55000.0,15.32,57.17,2. Mid-Low (300k-500k)
2,BMW,220i,1250000.0,1404000.0,164000.0,13.12,137.11,4. Premium (>1M)


## 📐 Section 2: คำนวณ Financial & Derived Performance Measures
คำนวณ 6 Performance Measures หลัก (`profit`, `profit_margin`, `discount_amount`, `discount_pct`, `depreciation_amount`, `discount_to_deprec_ratio`)

In [ ]:
np.random.seed(42)
df_trans['car_age'] = (2026 - df_trans['model_year']).clip(lower=1)
df_trans['list_price'] = (df_trans['price_clean'] * np.random.uniform(1.05, 1.15, size=len(df_trans))).round(-3)
df_trans['discount_amount'] = (df_trans['list_price'] - df_trans['price_clean']).round(2)
df_trans['discount_pct'] = ((df_trans['discount_amount'] / df_trans['list_price']) * 100).round(2)

df_trans['cost_price'] = (df_trans['price_clean'] * np.random.uniform(0.80, 0.88, size=len(df_trans))).round(-3)
df_trans['profit'] = (df_trans['price_clean'] - df_trans['cost_price']).round(2)
df_trans['profit_margin'] = ((df_trans['profit'] / df_trans['price_clean']) * 100).round(2)

deprec_rate = 0.08
df_trans['depreciation_amount'] = (df_trans['list_price'] * (1 - (1 - deprec_rate)**df_trans['car_age'])).round(2)
df_trans['discount_to_deprec_ratio'] = np.where(df_trans['depreciation_amount'] > 0, (df_trans['discount_amount'] / df_trans['depreciation_amount']) * 100, 0).round(2)
df_trans['days_on_lot'] = np.random.randint(10, 120, size=len(df_trans))
df_trans['net_revenue'] = df_trans['price_clean']

def assign_price_tier(p):
    if p < 300000: return '1. Eco (<300k)'
    elif p < 500000: return '2. Mid-Low (300k-500k)'
    elif p < 1000000: return '3. Mid-High (500k-1M)'
    else: return '4. Premium (>1M)'

df_trans['price_tier'] = df_trans['price_clean'].apply(assign_price_tier)
print('✅ คำนวณ Derived Financial Measures สำเร็จครบถ้วน')
df_trans[['brand', 'model', 'price_clean', 'list_price', 'profit', 'profit_margin', 'discount_to_deprec_ratio', 'price_tier']].head(5)